# Fixed-siRNA nucleolus circularity (Figure 5)

In the fixed siRNA endpoint (63x, Hoechst + NPM3-GFP), the three CCT/TRiC chaperonin-subunit knockdowns &mdash; **siCCT6A, siCCT3, siTCP1** &mdash; make nucleoli dramatically **rounder** than siControl (circularity &uarr;). Per-nucleolus circularity distributions, **cell-balanced** (per-cell subsampled to 3,371 cells/group), nucleoli with area &ge; 100 px.

Input data, committed alongside this notebook:

| file | rows | columns |
| --- | --- | --- |
| `nucleoli_circularity_percell.csv` | 52,612 nucleoli | `group` (siRNA condition), `well`, `area` (px), `circularity` |

`circularity = 4&pi;&middot;area / perimeter_crofton&sup2;`, clipped to [0,1] (1 = perfect circle). Significance = two-sided Mann&ndash;Whitney U vs pooled siControl (\*\*\* p&lt;1e-3).

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# Keep text editable in Illustrator (SVG keeps <text> elements; PDF uses TrueType).
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]

FIGURES_DIR = Path("../../output/figure_5")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Data

Per-nucleolus CSV committed alongside this notebook; filter to area &ge; 100 px.

In [ ]:
FIGURE_DATA = Path(".")
AREA_LO = 100
METRIC = "circularity"

df = pd.read_csv(FIGURE_DATA / "nucleoli_circularity_percell.csv")
d = df[df["area"] > AREA_LO]
print(f"{len(d):,} nucleoli after area>{AREA_LO} filter")

## Configuration

Condition order + display labels + colours (tan / blue / green / orange).

In [ ]:
ORDER  = ["siControl", "siCCT6A", "siCCT3", "siTCP1"]
LABELS = {"siControl": "Control", "siCCT6A": "CCT6A", "siCCT3": "CCT3A", "siTCP1": "TCP1"}
COLORS = {"siControl": "#c9a96b", "siCCT6A": "#6fb2d8", "siCCT3": "#93c973", "siTCP1": "#ef915a"}

## Figure 5 &mdash; nucleolus circularity violins

One violin per condition; black median bar; dashed reference line at the Control median; significance stars above each knockdown. Saves an SVG (paper) + PNG.

In [ ]:
data = [d.loc[d.group == g, METRIC].dropna().to_numpy() for g in ORDER]
ctrl = d.loc[d.group == "siControl", METRIC].dropna().to_numpy()
ctrl_med = float(np.median(ctrl))

fig, ax = plt.subplots(figsize=(5.0, 6.5))
vp = ax.violinplot(data, positions=range(len(ORDER)), widths=0.85, showextrema=False)
for body, g in zip(vp["bodies"], ORDER):
    body.set_facecolor(COLORS[g]); body.set_alpha(0.9)
    body.set_edgecolor("#4d4d4d"); body.set_linewidth(1.5)

# dashed reference line at the Control median
ax.axhline(ctrl_med, ls=(0, (5, 4)), color="0.6", lw=1.5, zorder=1)

for i, (g, arr) in enumerate(zip(ORDER, data)):
    ax.hlines(np.median(arr), i - 0.30, i + 0.30, color="black", lw=5, zorder=5)  # median bar
    if g != "siControl":
        _, p = mannwhitneyu(arr, ctrl, alternative="two-sided")
        stars = "n.s." if p >= 0.05 else "*" if p >= 1e-2 else "**" if p >= 1e-3 else "***"
        ax.text(i, 1.03, stars, ha="center", va="bottom", fontsize=22, fontweight="bold")

ax.set_xticks(range(len(ORDER)))
ax.set_xticklabels([LABELS[g] for g in ORDER], rotation=45, ha="right", fontsize=19)
ax.set_ylim(0.0, 1.12)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.tick_params(axis="y", labelsize=19, width=1.5, length=6)
ax.tick_params(axis="x", length=0)
ax.set_title("Nucleolus\ncircularity", fontsize=25, pad=12)
ax.set_xlabel("(siRNA)", loc="right", fontsize=19)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_linewidth(1.5)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "nucleoli_circularity_violin.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "nucleoli_circularity_violin.png", dpi=200, bbox_inches="tight")
plt.show()

## Summary table

Per-condition n / median circularity + effect vs siControl (Cliff's &delta;, Mann&ndash;Whitney p).

In [ ]:
rows = []
for g, arr in zip(ORDER, data):
    r = {"group": g, "n_nucleoli": int(arr.size),
         "circularity_median": float(np.median(arr)),
         "circularity_q25": float(np.percentile(arr, 25)),
         "circularity_q75": float(np.percentile(arr, 75))}
    if g != "siControl":
        U, p = mannwhitneyu(arr, ctrl, alternative="two-sided")
        r["cliffs_delta"] = float(2 * U / (arr.size * ctrl.size) - 1)
        r["mannwhitney_p"] = float(p)
    rows.append(r)

summary = pd.DataFrame(rows)
summary.to_csv(FIGURES_DIR / "nucleoli_circularity_violin_summary.csv", index=False)
summary